# Explore Collections in Corpus

This notebook explores the document collections in the metadata index, showing:
1. Distribution of documents by collection
2. Example text from different collections
3. Quality distribution by collection

In [ ]:
import pandas as pd
from pathlib import Path
import sys

# Add project root to path
PROJECT_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.post_training.config import get_paths

In [ ]:
# Load metadata index
period = "1950_1999"
paths = get_paths(period)

print(f"Loading metadata from: {paths['metadata_index']}")
meta = pd.read_parquet(paths['metadata_index'])
print(f"Total documents: {len(meta):,}")

## 1. Collection Distribution

In [ ]:
# Count documents per collection
collection_counts = meta['collection'].value_counts()

print(f"{'Collection':<30} {'Count':>12} {'Percentage':>10}")
print("=" * 55)

total = len(meta)
for collection, count in collection_counts.items():
    pct = count / total * 100
    print(f"{collection:<30} {count:>12,} {pct:>9.2f}%")

print("=" * 55)
print(f"{'TOTAL':<30} {total:>12,}")

## 2. Example Text from Collections

Let's retrieve actual text samples from various collections.

In [ ]:
# Get a sample of documents (one per collection)
raw_root = paths['raw_data_root']

# Sample one document per collection
samples = meta.groupby('collection').head(1)
print(f"Sampling {len(samples)} documents (1 per collection)")

In [ ]:
def get_document_text(row, raw_root):
    """Retrieve text for a document from raw parquet files."""
    parquet_path = raw_root / str(row['year']) / row['subset_file']
    if not parquet_path.exists():
        return f"[File not found: {parquet_path}]"
    
    df = pd.read_parquet(parquet_path, columns=['identifier', 'text'])
    match = df[df['identifier'] == row['identifier']]
    if len(match) == 0:
        return "[Document not found in parquet]"
    
    text = match.iloc[0]['text']
    return text if pd.notna(text) else "[Empty text]"


print("Retrieving text samples...\n")

for idx, row in samples.iterrows():
    print("=" * 70)
    print(f"Collection: {row['collection']}")
    print(f"Year: {row['year']}")
    print(f"Identifier: {row['identifier']}")
    print(f"Quality Score: {row['predicted_quality']:.2f}")
    print("-" * 70)
    
    text = get_document_text(row, raw_root)
    # Show first 1500 characters
    preview = text[:1500] + "..." if len(text) > 1500 else text
    print(preview)
    print("\n")

## 3. Quality Distribution by Collection

In [ ]:
# Quality stats per collection
print("Quality Score Statistics by Collection:\n")
print(f"{'Collection':<30} {'Count':>10} {'Mean':>8} {'Median':>8} {'Std':>8}")
print("-" * 70)

for collection in collection_counts.index:
    coll_data = meta[meta['collection'] == collection]['predicted_quality']
    print(f"{collection:<30} {len(coll_data):>10,} {coll_data.mean():>8.2f} {coll_data.median():>8.2f} {coll_data.std():>8.2f}")

In [ ]:
# Overall quality stats
print("\nOverall Quality Statistics:")
print(f"  Mean:   {meta['predicted_quality'].mean():.3f}")
print(f"  Median: {meta['predicted_quality'].median():.3f}")
print(f"  Std:    {meta['predicted_quality'].std():.3f}")
print(f"  Min:    {meta['predicted_quality'].min():.3f}")
print(f"  Max:    {meta['predicted_quality'].max():.3f}")

# Percentiles
print("\nQuality Percentiles:")
for p in [25, 50, 75, 90, 95]:
    val = meta['predicted_quality'].quantile(p/100)
    print(f"  {p}th percentile: {val:.3f}")

In [ ]:
# Optional: Plot if matplotlib is available
try:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart of collections
    ax1 = axes[0]
    collections = collection_counts.index.tolist()
    counts = collection_counts.values
    bars = ax1.barh(collections, counts, color='#3498db')
    ax1.set_xlabel('Number of Documents')
    ax1.set_title('Documents by Collection')
    ax1.invert_yaxis()
    
    # Quality distribution histogram
    ax2 = axes[1]
    ax2.hist(meta['predicted_quality'], bins=50, color='#2ecc71', edgecolor='black', alpha=0.7)
    ax2.axvline(meta['predicted_quality'].median(), color='red', linestyle='--', label=f'Median: {meta["predicted_quality"].median():.2f}')
    ax2.set_xlabel('Quality Score')
    ax2.set_ylabel('Count')
    ax2.set_title('Quality Score Distribution')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("matplotlib not available - skipping visualization")